## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Fall 2025
## Introduction to Spark

In this lab we will use **PySpark** to explore the MovieLens 100k dataset.  
By the end we should be able to:
- Start a SparkSession and understand *lazy* evaluation
- Load CSV/TSV with explicit schemas
- Join DataFrames and compute aggregations
- Inspect jobs and stages in the Spark UI

**Runtime:** ~15–20 min on Colab. **Data:** MovieLens 100k and 32m.

In [1]:
# Spark now comes installed by default on Google colab
!pyspark --version

Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.
The system cannot find the path specified.
The system cannot find the path specified.


In [ ]:
# Start Spark. We will always access spark through this varible.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("csce676-spark-demo")
    .master("local[*]")
    .getOrCreate()
)

In [ ]:
# A quick test to make sure Spark is running
from pyspark.sql import functions as F

spark.createDataFrame([(1, "foo"), (2, "bar"), (3, "baz")],["id", "dummy"]).show()

In [ ]:
# We want to view jobs using Spark UI.
!pip -q install pinggy

In [ ]:
import pinggy

# Start an HTTP tunnel forwarding traffic to localhost on Spark UI's default port (4040)
tunnel = pinggy.start_tunnel(forwardto="localhost:4040")
print(f"Tunnel started with URL: {tunnel.urls[-1]}")

In [ ]:
# Download MovieLens 100k
!wget -q --show-progress http://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q ml-100k.zip

!ln -s ml-100k/u.data .
!ln -s ml-100k/u.item .

In [ ]:
# Download MovieLens 32m

# !rm u.data u.item

# !wget -q --show-progress http://files.grouplens.org/datasets/movielens/ml-32m.zip -O ml-32m.zip
# !unzip -q ml-32m.zip

# !ln -s ml-32m/u.data .
# !ln -s ml-32m/u.item .

In [ ]:
# Load ratings into a dataframe (infer schema)
ratings = spark.read.csv("u.data", sep="\t")
ratings.show()

In [ ]:
# Load ratings into a dataframe (with schema)
schema_ratings = "user_id INT, movie_id INT, rating INT, timestamp LONG"
ratings = spark.read.csv("u.data", sep="\t", schema=schema_ratings)
ratings.show()

In [ ]:
# Load movies
schema_movies = "movie_id INT, title STRING, release_date STRING, video_release_date STRING, imdb_url STRING"
movies = spark.read.csv("u.item", sep="|", schema=schema_movies)
movies.show()

In [ ]:
# drop video_release_date adn imdb_url
movies = movies.drop("video_release_date", "imdb_url")
movies.show()

In [ ]:
# Basic stats
print("Number of ratings:", ratings.count())
print("Number of users:", ratings.select("user_id").distinct().count())
print("Number of movies:", ratings.select("movie_id").distinct().count())

ratings.describe(["rating"]).show()

In [ ]:
# Let's add movie titles to the ratings dataframe (i.e. join)
movielens = ratings.join(movies.select('movie_id','title'), "movie_id", 'left') # left -> we want to retain ratings if there corresponding movie doesn't have a title
movielens.show()
movielens.count()

In [ ]:
# plot ratings distribution
import seaborn as sns
import matplotlib.pyplot as plt

ratings_distribution_pd = ratings.groupBy("rating").count().orderBy("rating").toPandas() # see caution below!

plt.figure(figsize=(8, 6))
sns.barplot(x='rating', y='count', data=ratings_distribution_pd)
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

# Caution: Pandas dataframes still run on a single machine
We can convert a Spark dataframe to a Pandas dataframe but if you run this on a huge dataset the local machine may not be able to handle the size because while a Spark dataframe is distributed, a Pandas dataframe is not.

You can also do the opposite and convert a Pandas dataframe to a Spark dataframe (see the example below) which can be handy if you need to make some *smallish* local pandas dataframe available to other spark dataframes for example to use in joins.

In [ ]:
# Load a Pandas dataframe into Spark
import pandas as pd

# Create a Pandas DataFrame
pdf = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35]
})

# Convert Pandas DataFrame to Spark DataFrame
sdf = spark.createDataFrame(pdf)
sdf.show()

In [ ]:
# What are the most rated movies?
movielens.groupBy("title").count().orderBy("count", ascending=False).show(truncate=False)

In [ ]:
# What are the highest rated movies?
movielens.groupBy("title").avg("rating").orderBy("avg(rating)", ascending=False).show(truncate=False)

In [ ]:
# Show average rating a long side the number of ratings to gauge the average rating signficance
movielens.groupBy("title").agg(
    F.avg("rating").alias("avg_rating"),
    F.count("rating").alias("num_ratings")
).orderBy("avg_rating", ascending=False).show(truncate=False)

# Do you see a problem?!

Let's use IMBD formula:

\begin{align}
WR &= \frac{v}{v+m} \cdot R + \frac{m}{v+m} \cdot C \\
\text{where:} \quad
R &= \text{average rating for the movie} \\
v &= \text{number of ratings for the movie} \\
m &= \text{minimum number of ratings required to be considered} \\
C &= \text{mean rating across all movies}
\end{align}


In [ ]:
# 1) Per-title stats: v (#ratings) and R (mean rating)
title_stats = (
    movielens.groupBy("title")
    .agg(
        F.count("rating").alias("v"),
        F.avg("rating").alias("R")
    )
)

# 2) Global mean rating C (across all movies)
C = movielens.agg(F.avg("rating").alias("C")).first()["C"]

# 3) Choose m = minimum votes to be considered "established"
#    A common practice is to set m to a high quantile (e.g., 90th percentile) of vote counts.
m = title_stats.stat.approxQuantile("v", [0.90], 0.0)[0]

# 4) IMDb-style weighted rating:
#    WR = (v/(v+m)) * R + (m/(v+m)) * C
ranked = (
    title_stats
    .withColumn("WR", (F.col("v")/(F.col("v")+F.lit(m))) * F.col("R") + (F.lit(m)/(F.col("v")+F.lit(m))) * F.lit(C))
    # Optional: keep only titles with at least m ratings (IMDb does this)
    .filter(F.col("v") >= F.lit(m))
    .orderBy(F.col("WR").desc(), F.col("v").desc())
)

ranked.show(truncate=False)

# Bonus: let's do MapReduce in Spark
Hello World example of MapReduce is usually word count. Let's perform word count on movie titles using the MapReduce paradigm. We need to design our mapper and reducer functions.

In [ ]:
# Mapper: split title into words
def mapper(title: str):
    return title.split(" ")

# Reducer: sum counts
def reducer(a: int, b: int) -> int:
    return a + b

In [ ]:
titles_rdd = movies.select("title").rdd.map(lambda row: row.title)
titles_rdd.take(5)

In [ ]:
words_rdd = titles_rdd.flatMap(mapper) # .map function produces one output while *flatMap* can produce more than one output which suitable for our Mapper function
words_rdd.take(5)

In [ ]:
# Map each word to (word, 1)
pairs_rdd = words_rdd.map(lambda w: (w, 1))
pairs_rdd.take(5)

In [ ]:
# Reduce: sum counts
counts_rdd = pairs_rdd.reduceByKey(reducer)
counts_rdd.take(5)

In [ ]:
# Sort by count (descending)
sorted_counts = counts_rdd.map(lambda kv: (kv[1], kv[0])).sortByKey(ascending=False)
sorted_counts.take(20)

In [ ]:
# We can also do the same MapReduce-style code above directly in DataFrames like this
words_df = movies.select(F.col("title"))\
      .na.drop(subset=["title"])\
      .select(F.explode(F.split(F.col("title"), " ")).alias("word"))\
      .groupBy("word").count()\
      .orderBy(F.desc("count"))

words_df.show(20)

In [ ]:
# Print query execution plan
words_df.explain(True)


Excercises:
1. Find users who rated > 300 movies?
2. What year has the most releases?